# Target 2020用：窒素・リン投入量とGDPデータのダウンロード


このノートブックは、`cropland_soil_group_wetland_excluded_oof_shap_target2020.ipynb` のモデルを再計算せず、追加候補変数の元データだけをサーバーに取得するためのものです。


## データの位置づけ


- **窒素・リン**：Porfirio et al. (2017) と同じ SEDAC の *Global Fertilizer and Manure v1*（1994--2001の国別統計を作物面積へ空間配分、kg/ha）。

- **GDP**：同論文のGDPは厳密には18地域の正規化GDPであり、一人当たりGDP格子ではありません。ここでは、2020年を取り出せる公開5分格子のPPP一人当たりGDPを、`gdp_pc_2020_proxy` として別扱いで取得します。先行研究のGDP変数そのものと同一視しません。


この段階では、target 2020モデルの学習・OOF・SHAPは実行しません。ダウンロード、ファイル整合性確認、簡単なラスタQCだけを行います。


In [1]:
from pathlib import Path

import csv

import hashlib

import json

import re

import sys

from html.parser import HTMLParser

from urllib.parse import urljoin

from urllib.request import Request, urlopen



print(sys.version)

print('Python imports are ready.')


3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:46:43) [GCC 14.3.0]
Python imports are ready.


In [2]:
# サーバー上の保存先

SERVER_ROOT = Path('/work/tsuda')

GAEZ_DIR = SERVER_ROOT / 'GAEZ'

DOWNLOAD_DIR = GAEZ_DIR / 'CroplandRegression' / 'prior_study_inputs'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


# target 2020の5分グリッドと同じ場所に後で派生変数を保存するための場所

DERIVED_DIR = DOWNLOAD_DIR / 'derived_5arcmin_target2020'

DERIVED_DIR.mkdir(parents=True, exist_ok=True)


# SEDACのデータ配布ページ。公開ページから実ファイルのリンクを自動探索する。

SEDAC_DATASETS = {

    'nitrogen_fertilizer': {

        'landing_url': 'https://sedac.ciesin.columbia.edu/data/set/ferman-v1-nitrogen-fertilizer-application/data-download',

        'doi': 'https://doi.org/10.7927/H4Q81B0R',

        'keywords': ('nitrogen', 'fertilizer', 'ferman'),

    },

    'phosphorus_fertilizer': {

        'landing_url': 'https://sedac.ciesin.columbia.edu/data/set/ferman-v1-phosphorus-fertilizer-application/data-download',

        'doi': 'https://doi.org/10.7927/H4FQ9TJR',

        'keywords': ('phosphorus', 'fertilizer', 'ferman'),

    },

}


# SEDACのページが認証やJavaScriptで直接リンクを返さない場合だけ、

# ここに配布ページで確認した実ファイルURLを入力する。通常は空欄のままでよい。

SEDAC_DIRECT_URLS = {

    'nitrogen_fertilizer': '',

    'phosphorus_fertilizer': '',

}


# Kummu et al. の公開GDP per capita raster。1990--2022の各年をバンドに持ち、

# 2020年は31番目のバンド。144.5 MB程度で、全データ一括ではない。

GDP_PROXY_URL = (

    'https://zenodo.org/records/10976734/files/'

    'rast_adm2_gdp_pc_1990_2022.tif?download=1'

)

GDP_PROXY_MD5 = '1547c0dbf374c3753bfe65be9831acf7'

GDP_PROXY_PATH = DOWNLOAD_DIR / 'gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif'

MANIFEST_PATH = DOWNLOAD_DIR / 'download_manifest.csv'


print('DOWNLOAD_DIR:', DOWNLOAD_DIR)

print('GDP proxy path:', GDP_PROXY_PATH)


DOWNLOAD_DIR: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs
GDP proxy path: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif


In [5]:
%pip install -U earthaccess

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 104.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [earthaccess] [earthaccess]]ources]
Note: you may need to restart the kernel to use updated packages.


## 1. SEDACの窒素・リンを取得


SEDACの配布ページが実ファイルのリンクを返す場合は自動取得します。もし認証画面やHTMLだけが返った場合、このセルは「失敗」として止まります。その場合は、SEDAC配布ページから実ファイルURLを取得して、上の `SEDAC_DIRECT_URLS` に貼り、同じセルを再実行してください。


HTMLをGeoTIFFとして保存することはしません。


In [2]:
import earthaccess

auth = earthaccess.login(
    strategy="interactive",
    persist=True,
)

if not auth.authenticated:
    raise RuntimeError(
        "Earthdata Loginに失敗しました。"
    )

print("Earthdata Login succeeded.")

Earthdata Login succeeded.


In [3]:
import earthaccess

datasets = {
    "nitrogen": "CIESIN_SEDAC_FERMANv1_NAPP",
    "phosphorus": "CIESIN_SEDAC_FERMANv1_PAPP",
}

granules_by_dataset = {}

for name, short_name in datasets.items():
    granules = earthaccess.search_data(
        short_name=short_name,
        version="1.0",
        downloadable=True,
        count=-1,
    )

    granules_by_dataset[name] = granules

    print(f"\n{name}: {len(granules)} granules")

    for i, granule in enumerate(granules):
        print(f"\n[{i}]")

        for url in granule.data_links():
            print(url)


nitrogen: 14 granules

[0]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-root/downloads/data/ferman-v1/ferman-v1-nitrogen-fertilizer-application/nitrogen-fertilizer-global-geotif.zip

[1]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-root/downloads/data/ferman-v1/ferman-v1-nitrogen-fertilizer-application/nitrogen-fertilizer-africa-geotif.zip

[2]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-root/downloads/data/ferman-v1/ferman-v1-nitrogen-fertilizer-application/nitrogen-fertilizer-asia-geotif.zip

[3]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-root/downloads/data/ferman-v1/ferman-v1-nitrogen-fertilizer-application/nitrogen-fertilizer-europe-geotif.zip

[4]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-root/downloads/data/ferman-v1/ferman-v1-nitrogen-fertilizer-application/nitrogen-fertilizer-oceania-geotif.zip

[5]
https://data.earthdata.nasa.gov/nasa-earth/human-dimensions/sedac-ro

In [4]:
from pathlib import Path
import shutil
import zipfile

import earthaccess


DOWNLOAD_DIR = Path(
    "/work/tsuda/GAEZ/CroplandRegression/prior_study_inputs"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Earthdata Searchで取得したgranuleの[0]を指定
nitrogen_granule = granules_by_dataset["nitrogen"][0]
phosphorus_granule = granules_by_dataset["phosphorus"][0]


# 全球GeoTIFF ZIPをサーバーへ直接ダウンロード
earthaccess.download(
    [nitrogen_granule],
    local_path=str(DOWNLOAD_DIR),
)

earthaccess.download(
    [phosphorus_granule],
    local_path=str(DOWNLOAD_DIR),
)


print("ZIPファイルのダウンロード完了")

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

ZIPファイルのダウンロード完了


In [5]:
# ------------------------------------------------------------
# ZIPを展開し、モデル用のファイル名に統一
# ------------------------------------------------------------

nitrogen_zips = list(
    DOWNLOAD_DIR.rglob(
        "nitrogen-fertilizer-global-geotif.zip"
    )
)

phosphorus_zips = list(
    DOWNLOAD_DIR.rglob(
        "phosphorus-fertilizer-global-geotif.zip"
    )
)

if not nitrogen_zips:
    raise FileNotFoundError(
        "窒素のglobal-geotif.zipが見つかりません。"
    )

if not phosphorus_zips:
    raise FileNotFoundError(
        "リンのglobal-geotif.zipが見つかりません。"
    )


extract_jobs = [
    (
        nitrogen_zips[0],
        DOWNLOAD_DIR / "ferman_nitrogen_extracted",
        DOWNLOAD_DIR / "nitrogen_fertilizer__ferman_v1_global.tif",
    ),
    (
        phosphorus_zips[0],
        DOWNLOAD_DIR / "ferman_phosphorus_extracted",
        DOWNLOAD_DIR / "phosphorus_fertilizer__ferman_v1_global.tif",
    ),
]


for zip_path, extract_dir, output_path in extract_jobs:
    extract_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"\n展開中: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(extract_dir)

    tif_candidates = [
        path
        for path in extract_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".tif", ".tiff"}
    ]

    if not tif_candidates:
        raise FileNotFoundError(
            f"GeoTIFFが見つかりません: {extract_dir}"
        )

    print("GeoTIFF候補:")
    for path in tif_candidates:
        print(" ", path)

    # 全球GeoTIFFをモデル用の固定名で保存
    shutil.copy2(
        tif_candidates[0],
        output_path,
    )

    print(f"保存完了: {output_path}")
    print(
        f"サイズ: "
        f"{output_path.stat().st_size:,} bytes"
    )


print("\n窒素・リンのFERMAN v1全球GeoTIFF準備完了")


展開中: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/nitrogen-fertilizer-global-geotif.zip
GeoTIFF候補:
  /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/ferman_nitrogen_extracted/NFertilizer_global_geotif/nfertilizer_global.tif
保存完了: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/nitrogen_fertilizer__ferman_v1_global.tif
サイズ: 805,616 bytes

展開中: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/phosphorus-fertilizer-global-geotif.zip
GeoTIFF候補:
  /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/ferman_phosphorus_extracted/PFertilizer_global_geotif/pfertilizer_global.tif
保存完了: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/phosphorus_fertilizer__ferman_v1_global.tif
サイズ: 805,992 bytes

窒素・リンのFERMAN v1全球GeoTIFF準備完了


## 2. 2020年の一人当たりGDPプロキシを取得


このファイルはKummu et al. の公開データで、PPP・2017年国際ドルの一人当たりGDPです。1990--2022年の年次バンドを持つため、2020年バンドを後で5分グリッドへ合わせます。全ファイル一括ではなく、約145 MBのGDP per capita rasterだけを取得します。


これはPorfirio et al. の18地域正規化GDPの再現ではありません。論文どおりのGDPを再現する場合は、GTEM-Cの地域GDPシナリオ値を別途入手して、別変数として扱う必要があります。


In [11]:
# ============================================================
# Section 2
# 2020年 GDP per capita プロキシのダウンロード
# ============================================================

from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import csv
import time


# ------------------------------------------------------------
# 1. 保存先とデータ設定
# ------------------------------------------------------------

DOWNLOAD_DIR = Path(
    "/work/tsuda/GAEZ/CroplandRegression/prior_study_inputs"
)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

GDP_PROXY_URL = (
    "https://zenodo.org/records/10976734/files/"
    "rast_adm2_gdp_pc_1990_2022.tif?download=1"
)

GDP_PROXY_MD5 = (
    "1547c0dbf374c3753bfe65be9831acf7"
)

GDP_PROXY_PATH = (
    DOWNLOAD_DIR
    / "gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif"
)

MANIFEST_PATH = (
    DOWNLOAD_DIR / "download_manifest.csv"
)

HTTP_TIMEOUT = 600
HTTP_RETRIES = 3
RETRY_WAIT = 10

print("保存先:", GDP_PROXY_PATH)


# ------------------------------------------------------------
# 2. MD5確認
# ------------------------------------------------------------

def calculate_md5(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.md5()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. GDPファイルのダウンロード
# ------------------------------------------------------------

def download_stream(
    url,
    destination,
    expected_md5=None,
):
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # 既存ファイルがあり、MD5が一致する場合は再利用
    if destination.exists():

        if expected_md5 is None:
            print("既存ファイルを再利用します:")
            print(destination)
            return destination

        existing_md5 = calculate_md5(destination)

        if existing_md5 == expected_md5:
            print("既存ファイルのMD5確認済み:")
            print(existing_md5)
            print("再ダウンロードしません。")
            return destination

        print("既存ファイルのMD5が不一致です。")
        print("再ダウンロードします。")

    last_error = None

    for attempt in range(1, HTTP_RETRIES + 1):

        try:
            print(
                f"\nGDPダウンロード "
                f"{attempt}/{HTTP_RETRIES}"
            )
            print(url)

            request = Request(
                url,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "research-data-downloader"
                    )
                },
            )

            with urlopen(
                request,
                timeout=HTTP_TIMEOUT,
            ) as response:

                content_type = (
                    response.headers.get(
                        "Content-Type"
                    )
                    or ""
                ).lower()

                print("Content-Type:", content_type)
                print(
                    "Content-Length:",
                    response.headers.get(
                        "Content-Length"
                    ),
                )

                first_chunk = response.read(1024)

                # HTMLが返った場合はエラー
                if (
                    "text/html" in content_type
                    or first_chunk.lstrip().startswith(b"<")
                ):
                    preview = first_chunk.decode(
                        "utf-8",
                        errors="replace",
                    )[:300]

                    raise RuntimeError(
                        "GDPデータではなくHTMLが返されました。"
                        f"\nPreview: {preview}"
                    )

                temporary_path = destination.with_suffix(
                    destination.suffix + ".part"
                )

                total_bytes = len(first_chunk)

                with temporary_path.open("wb") as handle:
                    handle.write(first_chunk)

                    while True:
                        chunk = response.read(
                            8 * 1024 * 1024
                        )

                        if not chunk:
                            break

                        handle.write(chunk)
                        total_bytes += len(chunk)

                        if (
                            total_bytes
                            % (100 * 1024 * 1024)
                            < 8 * 1024 * 1024
                        ):
                            print(
                                "取得済み:",
                                f"{total_bytes / 1024**2:.1f} MB",
                            )

                temporary_path.replace(destination)

            print("ダウンロード完了:")
            print(destination)

            break

        except Exception as error:
            last_error = error

            print(
                "ダウンロード失敗:",
                type(error).__name__,
                error,
            )

            if attempt < HTTP_RETRIES:
                print(
                    f"{RETRY_WAIT}秒後に再試行します。"
                )
                time.sleep(RETRY_WAIT)

    else:
        raise RuntimeError(
            "GDPファイルのダウンロードに失敗しました。"
        ) from last_error

    # MD5確認
    if expected_md5 is not None:

        actual_md5 = calculate_md5(destination)

        print("期待MD5:", expected_md5)
        print("実際MD5:", actual_md5)

        if actual_md5 != expected_md5:
            raise RuntimeError(
                "MD5が一致しません。"
                "ファイルが途中で切れた可能性があります。"
            )

        print("MD5確認完了。")

    print(
        "ファイルサイズ:",
        f"{destination.stat().st_size / 1024**2:.2f} MB",
    )

    return destination


# ------------------------------------------------------------
# 4. マニフェストへの記録
# ------------------------------------------------------------

def append_manifest(rows):
    """
    辞書1件でも、辞書のリストでも受け付ける。
    """

    if isinstance(rows, dict):
        rows = [rows]

    fieldnames = [
        "dataset",
        "source_url",
        "local_path",
        "status",
        "notes",
    ]

    write_header = not MANIFEST_PATH.exists()

    with MANIFEST_PATH.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
        )

        if write_header:
            writer.writeheader()

        writer.writerows(rows)


# ------------------------------------------------------------
# 5. GDPデータ取得
# ------------------------------------------------------------

gdp_path = download_stream(
    GDP_PROXY_URL,
    GDP_PROXY_PATH,
    expected_md5=GDP_PROXY_MD5,
)

append_manifest({
    "dataset": "gdp_pc_2020_proxy",
    "source_url": GDP_PROXY_URL,
    "local_path": str(gdp_path),
    "status": "downloaded_or_reused",
    "notes": (
        "Kummu gridded GDP per capita PPP; "
        "1990-2022; 2020年バンドを使用"
    ),
})

print("\nGDPデータ取得完了。")
print("保存先:", gdp_path)


# ------------------------------------------------------------
# 6. ダウンロード結果確認
# ------------------------------------------------------------

print("\n保存ファイル一覧:")

for path in sorted(DOWNLOAD_DIR.iterdir()):

    if path.is_file():
        print(
            f"{path.name}: "
            f"{path.stat().st_size / 1024**2:.2f} MB"
        )

print("\nマニフェスト:")
print(MANIFEST_PATH)

if MANIFEST_PATH.exists():
    print(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )


# ------------------------------------------------------------
# 7. GDPラスタのメタデータ確認
# ------------------------------------------------------------

try:
    import rasterio

except ImportError:
    print(
        "rasterioがないため、"
        "ラスタメタデータ確認をスキップします。"
    )

else:

    with rasterio.open(gdp_path) as src:

        print("\nGDPラスタ情報")
        print("CRS:", src.crs)
        print("幅:", src.width)
        print("高さ:", src.height)
        print("解像度:", src.res)
        print("バンド数:", src.count)
        print("範囲:", src.bounds)

        # 1990年を1バンド目とすると、
        # 2020年は31バンド目
        expected_2020_band = 2020 - 1990 + 1

        print(
            "2020年バンド:",
            expected_2020_band,
        )

        if src.count < expected_2020_band:
            raise RuntimeError(
                "2020年のバンドが存在しません。"
            )

        print("2020年バンドの存在を確認しました。")

保存先: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif
既存ファイルのMD5確認済み:
1547c0dbf374c3753bfe65be9831acf7
再ダウンロードしません。

GDPデータ取得完了。
保存先: /work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif

保存ファイル一覧:
download_manifest.csv: 0.00 MB
gdp_pc_ppp_2017usd_5arcmin_1990_2022.tif: 137.83 MB

マニフェスト:
/work/tsuda/GAEZ/CroplandRegression/prior_study_inputs/download_manifest.csv
dataset,source_url,local_path,status,notes
nitrogen_fertilizer,https://sedac.ciesin.columbia.edu/data/set/ferman-v1-nitrogen-fertilizer-application/data-download,,failed,RuntimeError('URLへ接続できませんでした: https://sedac.ciesin.columbia.edu/data/set/ferman-v1-nitrogen-fertilizer-application/data-download')
phosphorus_fertilizer,https://sedac.ciesin.columbia.edu/data/set/ferman-v1-phosphorus-fertilizer-application/data-download,,failed,RuntimeError('URLへ接続できませんでした: https://sedac.ciesin.columbia.edu/data/set/ferman-v1-phosphorus-fertilizer-applic

## 次の段階


1. SEDACのN/Pファイル形式と座標系を確認する。

2. 0.5度のN/Pをtarget 2020の5分グリッドへ再投影・再サンプリングする。

3. GDP per capitaの2020年バンドを抽出して5分グリッドへ合わせる。

4. 既存モデルの入力変数に追加した比較版を作る。

5. baseline / +N&P / +GDP / +N&P+GDPを同じOOF分割で比較する。


このノートブック自体では、4と5は行いません。


## 非都市人口ダウンロード

In [1]:
from pathlib import Path
import json
import requests
from urllib.parse import urlparse

# 保存先
DOWNLOAD_DIR = Path("/work/tsuda/GAEZ/PopulationData/rural_population_2020")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Zenodoレコード
RECORD_ID = "14246584"
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"

# ファイル一覧を取得
record = requests.get(API_URL, timeout=60)
record.raise_for_status()
record_json = record.json()

files = record_json["files"]

# 2020年の農村人口GeoTIFFを検索
candidates = [
    f for f in files
    if "rural.pop.count" in f["key"].lower()
    and "2020" in f["key"]
    and f["key"].lower().endswith((".tif", ".tiff"))
]

if not candidates:
    print("候補ファイル一覧:")
    for f in files:
        print(f["key"])
    raise RuntimeError("2020年の農村人口GeoTIFFが見つかりませんでした。")

print("候補ファイル:")
for f in candidates:
    print(f["key"], f["size"], "bytes")

# 通常は2020年ファイルを1つ選択
target_file = candidates[0]
download_url = target_file["links"]["self"]
destination = DOWNLOAD_DIR / target_file["key"]

print("ダウンロード対象:")
print(download_url)
print("保存先:")
print(destination)

# ダウンロード
if destination.exists() and destination.stat().st_size == target_file["size"]:
    print("既存ファイルを再利用します。")
else:
    with requests.get(download_url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        downloaded = 0

        with open(destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total:
                        print(
                            f"\r進捗: {downloaded / total:.1%}",
                            end=""
                        )

    print("\nダウンロード完了。")

print("保存済みファイル:")
print(destination)
print("サイズ:", destination.stat().st_size, "bytes")

候補ファイル:
rural.pop.count_worldpop_m_1km_20200101_20201231_go_epsg4326_v20241129.tif 94717285 bytes
rural.pop.count_worldpop_m_5km_20200101_20201231_go_epsg4326_v20241129.tif 5180369 bytes
ダウンロード対象:
https://zenodo.org/api/records/14246584/files/rural.pop.count_worldpop_m_1km_20200101_20201231_go_epsg4326_v20241129.tif/content
保存先:
/work/tsuda/GAEZ/PopulationData/rural_population_2020/rural.pop.count_worldpop_m_1km_20200101_20201231_go_epsg4326_v20241129.tif
進捗: 100.0%
ダウンロード完了。
保存済みファイル:
/work/tsuda/GAEZ/PopulationData/rural_population_2020/rural.pop.count_worldpop_m_1km_20200101_20201231_go_epsg4326_v20241129.tif
サイズ: 94717285 bytes


In [2]:
from pathlib import Path
import json
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from affine import Affine

# ============================================================
# パス
# ============================================================

DOWNLOAD_DIR = Path(
    "/work/tsuda/GAEZ/PopulationData/rural_population_2020"
)

OUTPUT_DIR = Path(
    "/work/tsuda/GAEZ/PopulationData/rural_population_2020_5min"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2020年の農村人口GeoTIFFを自動検出
input_files = sorted(DOWNLOAD_DIR.glob("*.tif")) + \
              sorted(DOWNLOAD_DIR.glob("*.tiff"))

if not input_files:
    raise FileNotFoundError(
        f"入力GeoTIFFが見つかりません: {DOWNLOAD_DIR}"
    )

INPUT_TIF = input_files[0]

print("入力ファイル:")
print(INPUT_TIF)
print("サイズ:", INPUT_TIF.stat().st_size, "bytes")


# ============================================================
# 5分グリッド設定
# ============================================================

N_LAT = 2160
N_LON = 4320

RESOLUTION = 1 / 12  # 5 arc-minutes

DST_TRANSFORM = Affine(
    RESOLUTION,
    0,
    -180,
    0,
    -RESOLUTION,
    90,
)

DST_CRS = "EPSG:4326"


# ============================================================
# 1 km人口を5分グリッドへ集計
# ============================================================

rural_population_5min = np.zeros(
    (N_LAT, N_LON),
    dtype=np.float64,
)

with rasterio.open(INPUT_TIF) as src:
    print("source CRS:", src.crs)
    print("source shape:", src.height, src.width)
    print("source bounds:", src.bounds)
    print("source nodata:", src.nodata)

    source_array = src.read(1).astype(np.float64)

    # nodataを0にする
    if src.nodata is not None:
        source_array[source_array == src.nodata] = 0.0

    source_array[~np.isfinite(source_array)] = 0.0
    source_array[source_array < 0] = 0.0

    reproject(
        source=source_array,
        destination=rural_population_5min,
        src_transform=src.transform,
        src_crs=src.crs if src.crs is not None else "EPSG:4326",
        dst_transform=DST_TRANSFORM,
        dst_crs=DST_CRS,
        resampling=Resampling.sum,
        src_nodata=0.0,
        dst_nodata=0.0,
    )

rural_population_5min[
    ~np.isfinite(rural_population_5min)
] = 0.0

rural_population_5min[
    rural_population_5min < 0
] = 0.0


# ============================================================
# 5分グリッドのセル面積を計算
# ============================================================

EARTH_RADIUS_KM = 6371.0088

lat_edges = np.linspace(90, -90, N_LAT + 1)
lat_north = lat_edges[:-1]
lat_south = lat_edges[1:]

lon_width_rad = np.deg2rad(RESOLUTION)

cell_area_km2_by_lat = (
    EARTH_RADIUS_KM ** 2
    * lon_width_rad
    * (
        np.sin(np.deg2rad(lat_north))
        - np.sin(np.deg2rad(lat_south))
    )
)

cell_area_km2 = np.repeat(
    cell_area_km2_by_lat[:, None],
    N_LON,
    axis=1,
)

# 非都市人口密度：人 / km²
rural_population_density_5min = np.divide(
    rural_population_5min,
    cell_area_km2,
    out=np.zeros_like(rural_population_5min),
    where=cell_area_km2 > 0,
)


# ============================================================
# 保存
# ============================================================

POP_OUTPUT = OUTPUT_DIR / "rural_population_2020_5min.npy"
DENSITY_OUTPUT = OUTPUT_DIR / "rural_population_density_2020_5min.npy"
AREA_OUTPUT = OUTPUT_DIR / "cell_area_km2_5min.npy"
METADATA_OUTPUT = OUTPUT_DIR / "metadata.json"

np.save(POP_OUTPUT, rural_population_5min.astype(np.float32))
np.save(
    DENSITY_OUTPUT,
    rural_population_density_5min.astype(np.float32),
)
np.save(AREA_OUTPUT, cell_area_km2.astype(np.float32))

metadata = {
    "input_file": str(INPUT_TIF),
    "year": 2020,
    "grid": "global_5_arcmin",
    "n_lat": N_LAT,
    "n_lon": N_LON,
    "resolution_degree": RESOLUTION,
    "crs": "EPSG:4326",
    "population_unit": "persons_per_5arcmin_cell",
    "density_unit": "persons_per_km2",
    "description": (
        "Rural/non-urban population aggregated from approximately "
        "1-km raster to global 5-arcminute grid."
    ),
}

with open(METADATA_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)


# ============================================================
# QC
# ============================================================

print("\n完了しました。")
print("人口配列:", POP_OUTPUT)
print("人口密度:", DENSITY_OUTPUT)
print("セル面積:", AREA_OUTPUT)
print("メタデータ:", METADATA_OUTPUT)

print("\n配列shape:")
print("population:", rural_population_5min.shape)
print("density:", rural_population_density_5min.shape)

print("\n基本統計:")
print("total rural population:",
      float(np.nansum(rural_population_5min)))
print("max rural population per cell:",
      float(np.nanmax(rural_population_5min)))
print("max rural population density:",
      float(np.nanmax(rural_population_density_5min)))
print("non-zero cells:",
      int(np.count_nonzero(rural_population_5min)))

入力ファイル:
/work/tsuda/GAEZ/PopulationData/rural_population_2020/rural.pop.count_worldpop_m_1km_20200101_20201231_go_epsg4326_v20241129.tif
サイズ: 94717285 bytes
source CRS: EPSG:4326
source shape: 18720 43200
source bounds: BoundingBox(left=-180.001249265, bottom=-72.00041617728999, right=179.99874929500004, top=83.99958319871001)
source nodata: 0.0

完了しました。
人口配列: /work/tsuda/GAEZ/PopulationData/rural_population_2020_5min/rural_population_2020_5min.npy
人口密度: /work/tsuda/GAEZ/PopulationData/rural_population_2020_5min/rural_population_density_2020_5min.npy
セル面積: /work/tsuda/GAEZ/PopulationData/rural_population_2020_5min/cell_area_km2_5min.npy
メタデータ: /work/tsuda/GAEZ/PopulationData/rural_population_2020_5min/metadata.json

配列shape:
population: (2160, 4320)
density: (2160, 4320)

基本統計:
total rural population: 3351782490.6808033
max rural population per cell: 704072.9345129199
max rural population density: 11439.458403423001
non-zero cells: 1159982
